## Code to prepare nurse validation

In [1]:
pip install openpyxl


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- -----------------

In [ ]:
import pandas as pd
import random
import shutil
import zipfile
from pathlib import Path


# CONFIGURACIÓN


csv_path = Path(r"D:\Documentos\UPC\GUTTMANN\demo2\data\processed\dataset_upp_clean.csv")
output_dir = Path(r"D:\Documentos\UPC\GUTTMANN\demo2\data\nurse_validation")

random_seed = 42
n_nurses = 17

# 3 enfermeras con 300, 3 con 200, resto balanceado
target_counts = [300, 300, 300, 200, 200, 200]
total_reviews = 1930 * 2
remaining = total_reviews - sum(target_counts)
remaining_nurses = n_nurses - len(target_counts)

base = remaining // remaining_nurses
extra = remaining % remaining_nurses
target_counts += [base + 1 if i < extra else base for i in range(remaining_nurses)]

nurse_names = [f"nurse_{i:02d}" for i in range(1, n_nurses + 1)]


# CARGAR CSV


df = pd.read_csv(csv_path)

# Comprobaciones básicas
assert "image_path" in df.columns, "El CSV debe tener una columna llamada image_path"
assert "image_filename" in df.columns, "El CSV debe tener una columna llamada image_filename"

df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

print("Número de imágenes:", len(df))
print("Total revisiones:", len(df) * 2)
print("Targets:", dict(zip(nurse_names, target_counts)))
print("Suma targets:", sum(target_counts))


# ASIGNACIÓN RANDOM BALANCEADA


random.seed(random_seed)

assignments = {nurse: [] for nurse in nurse_names}
remaining_capacity = dict(zip(nurse_names, target_counts))

for _, row in df.iterrows():
    available = [n for n in nurse_names if remaining_capacity[n] > 0]
    
    # Escoger 2 enfermeras distintas, priorizando las que aún tienen más capacidad
    weights = [remaining_capacity[n] for n in available]
    nurse_1 = random.choices(available, weights=weights, k=1)[0]
    
    available_2 = [n for n in available if n != nurse_1 and remaining_capacity[n] > 0]
    weights_2 = [remaining_capacity[n] for n in available_2]
    nurse_2 = random.choices(available_2, weights=weights_2, k=1)[0]
    
    for nurse in [nurse_1, nurse_2]:
        assignments[nurse].append(row.to_dict())
        remaining_capacity[nurse] -= 1

print("Capacidad restante:", remaining_capacity)


# CREAR CARPETAS, EXCELS Y ZIPS


output_dir.mkdir(parents=True, exist_ok=True)

master_rows = []

for nurse in nurse_names:
    nurse_dir = output_dir / nurse
    img_dir = nurse_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    
    nurse_records = []
    
    for i, row in enumerate(assignments[nurse], start=1):
        src = Path(row["image_path"])
        ext = src.suffix.lower()
        new_name = f"{i:04d}{ext}"
        dst = img_dir / new_name
        
        shutil.copy2(src, dst)
        
        nurse_records.append({
            "image_number": f"{i:04d}",
            "stage_1_to_5": "",
            "comments": ""
        })
        
        master_rows.append({
            "nurse": nurse,
            "image_number_for_nurse": f"{i:04d}",
            "assigned_filename": new_name,
            "original_filename": row["image_filename"],
            "original_path": row["image_path"],
            "patient_id": row.get("patient_id", ""),
            "original_stage_if_available": row.get("stage", "")
        })
    
    # Excel para la enfermera
    nurse_excel = nurse_dir / f"{nurse}_validation_template.xlsx"
    pd.DataFrame(nurse_records).to_excel(nurse_excel, index=False)
    
    # ZIP
    zip_path = output_dir / f"{nurse}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file in nurse_dir.rglob("*"):
            zipf.write(file, file.relative_to(nurse_dir))
    
    print(f"{nurse}: {len(nurse_records)} imágenes -> {zip_path}")


# MASTER ADMINISTRATIVO


master_df = pd.DataFrame(master_rows)
master_df.to_excel(output_dir / "MASTER_assignments.xlsx", index=False)

print("Hecho.")

Número de imágenes: 1930
Total revisiones: 3860
Targets: {'nurse_01': 300, 'nurse_02': 300, 'nurse_03': 300, 'nurse_04': 200, 'nurse_05': 200, 'nurse_06': 200, 'nurse_07': 215, 'nurse_08': 215, 'nurse_09': 215, 'nurse_10': 215, 'nurse_11': 215, 'nurse_12': 215, 'nurse_13': 214, 'nurse_14': 214, 'nurse_15': 214, 'nurse_16': 214, 'nurse_17': 214}
Suma targets: 3860
Capacidad restante: {'nurse_01': 0, 'nurse_02': 0, 'nurse_03': 0, 'nurse_04': 0, 'nurse_05': 0, 'nurse_06': 0, 'nurse_07': 0, 'nurse_08': 0, 'nurse_09': 0, 'nurse_10': 0, 'nurse_11': 0, 'nurse_12': 0, 'nurse_13': 0, 'nurse_14': 0, 'nurse_15': 0, 'nurse_16': 0, 'nurse_17': 0}
nurse_01: 300 imágenes -> D:\Documentos\UPC\GUTTMANN\demo2\data\nurse_validation\nurse_01.zip
nurse_02: 300 imágenes -> D:\Documentos\UPC\GUTTMANN\demo2\data\nurse_validation\nurse_02.zip
nurse_03: 300 imágenes -> D:\Documentos\UPC\GUTTMANN\demo2\data\nurse_validation\nurse_03.zip
nurse_04: 200 imágenes -> D:\Documentos\UPC\GUTTMANN\demo2\data\nurse_valida